```c
//P11960 [GESP202503 五级] 平均分配 贪心算法
#include<iostream>
#include<vector>
#include<algorithm>
using namespace std;
//定义结构体存储每件物品的信息
struct Item{
    long long b;    //小B的价格
    long long c;    //小C的价格
    long long diff; //差价 b - c
};
//自定义比较函数
bool compareItems(const Item& x,const Item& y){
    return x.diff>y.diff;//按差价从大到小排序
}
int main(){
    //输入
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int n;
    if(!(cin>>n)) return 0;
    int totalItems=2*n;
    vector<Item> items(totalItems);
    for(int i=0;i<totalItems;++i){
        cin>>items[i].b;
    }
    for(int i=0;i<totalItems;++i){
        cin>>items[i].c;
        items[i].diff=items[i].b-items[i].c;
    }
    //计算
    sort(items.begin(),items.end(),compareItems);
    long long maxIncome=0;
    for(int i=0;i<n;++i){          //前n件物品差价大，适合卖给B
        maxIncome+=items[i].b;
    }
    for(int i=n;i<totalItems;++i){ //后n件物品差价小，适合卖给C
        maxIncome+=items[i].c;
    }
    //输出
    cout<<maxIncome<<endl;
    return 0;
}
```

```c
//P11961 [GESP202503 五级] 原根判断
#include<iostream>
#include<vector>
#include<cmath>
using namespace std;
typedef long long ll;                                      //定义长整型
ll quick_pow(ll base,ll exp,ll mod){                       //快速幂算法：计算(base^exp) % mod
    ll result=1;
    base%=mod;                                             //确保base在模范围内
    while(exp>0){
        if(exp&1){                                         //如果指数是奇数
            result=(result*base)%mod;
        }
        base=(base*base)%mod;                              //底数平方
        exp>>=1;                                           //指数右移一位（除以2）
    }
    return result;
}
vector<ll> get_prime_factors(ll n){                        //获取n的所有不同质因子
    vector<ll> factors;
    if(n%2==0){                                            //处理因子2
        factors.push_back(2);
        while(n%2==0){
            n/=2;
        }
    }
    for(ll i=3;i*i<=n;i+=2){                               //处理奇数因子，从3开始，步长为2
        if(n%i==0){
            factors.push_back(i);
            while(n%i==0){
                n/=i;
            }
        }
    }
    if(n>1){                                               //如果n仍然大于1，说明剩下的n本身是一个质因子
        factors.push_back(n);
    }
    return factors;
}
int main(){
    //输入
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    int T;
    if(!(cin>>T)) return 0;
    while(T--){
        ll a,p;
        cin>>a>>p;
        //计算
        ll phi=p-1;                                        //因为p是质数，φ(p) = p - 1
        vector<ll> prime_factors=get_prime_factors(phi);   //获取p-1的所有不同质因子
        bool is_primitive_root=true;
        for(ll q:prime_factors){                           //验证关键条件
            ll exponent=phi/q;
            ll val=quick_pow(a,exponent,p);                //计算a^((p-1)/q) mod p
            if(val==1){                                    //如果结果为1，说明a的阶整除(p-1)/q，即阶小于 p-1，不是原根
                is_primitive_root=false;
                break;
            }
        }
        //输出
        if (is_primitive_root) {
            cout << "Yes" << endl;
        } else {
            cout << "No" << endl;
        }
    }
    return 0;
}
```

```c
//P6091 【模板】原根
#include<iostream>
#include<vector>
#include<algorithm>
#include<cmath>
using namespace std;
typedef long long ll;
const int MAXN=1000005;
int primes[MAXN];                                                         //存储素数
int phi[MAXN];                                                            //存储欧拉函数值
bool is_composite[MAXN];                                                  //标记合数
bool has_root[MAXN];                                                      //标记是否有原根
int prime_cnt=0;
ll quick_pow(ll base,ll exp,ll mod){                                      //快速幂: (base^exp) % mod
    ll res=1;
    base%=mod;
    while(exp>0){
        if(exp&1){
            res=(res*base)%mod;
        }
        base=(base*base)%mod;
        exp>>=1;
    }
    return res;
}
ll gcd(ll a,ll b){                                                        //最大公约数
    return b==0?a:gcd(b,a%b);
}
void precompute(){                                                        //预处理素数、欧拉函数、是否有原根
    phi[1]=1;
    for(int i=2;i<MAXN;++i){                                              //线性筛求素数和欧拉函数
        if(!is_composite[i]){
            primes[++prime_cnt]=i;
            phi[i]=i-1;
        }
        for(int j=1;j<=prime_cnt&&(ll)i*primes[j]<MAXN;++j){
            is_composite[i*primes[j]]=true;
            if(i%primes[j]==0){
                phi[i*primes[j]]=phi[i]*primes[j];
                break;
            }else{
                phi[i*primes[j]]=phi[i]*(primes[j]-1);
            }
        }
    }
    has_root[2]=true;                                                     //2有原根
    has_root[4]=true;                                                     //4有原根
    for(int i=2;i<=prime_cnt;++i){                                        //我们需要奇素数，所以从i=2开始
        ll p=primes[i]; 
        ll pk=p;
        while(pk<MAXN){
            has_root[pk]=true;                                            //p^k(p为奇素数)有原根
            if(2*pk<MAXN){
                has_root[2*pk]=true;                                      //2*p^k(p为奇素数)有原根
            }
            if(pk>MAXN/p) break;                                          //防止溢出
            pk*=p;
        }
    }
}
vector<ll> get_prime_factors(ll n){                                       //获取n的所有不同质因子
    vector<ll> factors;
    for(int i=1;i<=prime_cnt&&(ll)primes[i]*primes[i]<=n;++i){
        if(n%primes[i]==0){
            factors.push_back(primes[i]);
            while(n%primes[i]==0){
                n/=primes[i];
            }
        }
    }
    if(n>1){
        factors.push_back(n);
    }
    return factors;
}
bool is_primitive_root(ll g,ll n,ll phi_n,const vector<ll>&phi_factors){  //判断g是否是模n的原根
    if(gcd(g,n)!=1) return false;
    if(phi_n==1) return true;                                             //特殊情况即n=2，此时没有质因子，循环不执行，直接返回true
    for(ll q:phi_factors){
        if(quick_pow(g,phi_n/q,n)==1){                                    //如果g^(phi_n/q)≡1(mod n)，则g不是原根
            return false;
        }
    }
    return true;
}
int main(){
    //输入
    ios::sync_with_stdio(false);
    cin.tie(nullptr);
    precompute();
    int T;
    if(!(cin>>T)) return 0;
    while(T--){
        ll n,d;
        cin>>n>>d;
        // 1. 判断是否有原根
        if(!has_root[n]){
            cout<<0<<"\n\n";
            continue;
        }
        ll phi_n=phi[n];
        // 2. 分解 phi(n) 的质因子，用于判定原根
        vector<ll> phi_factors=get_prime_factors(phi_n);
        // 3. 寻找最小原根 g
        ll g_min=-1;
        for(ll g=1;g<n;++g){                                              //从1开始枚举，因为n=2时原根是1
            if(is_primitive_root(g,n,phi_n,phi_factors)){
                g_min=g;
                break;
            }
        }
        // 4. 生成所有原根
        vector<ll> roots;
        ll current_pow=1;
        for(ll k=1;k<=phi_n;++k){
            current_pow=(current_pow*g_min)%n;
            if(gcd(k,phi_n)==1){
                roots.push_back(current_pow);
            }
        }
        // 5. 排序
        sort(roots.begin(),roots.end());
        //输出
        int count=roots.size();
        cout<<count<<"\n";
        for(int i=1;i*d<=count;++i){                                      //数组下标从0开始，所以第i*d个元素下标是i*d-1
            cout<<roots[i*d-1]<<" ";
        }
        cout<<"\n";
    }
    return 0;
}
```

```c
//P13013 [GESP202506 五级] 奖品兑换 二分答案
#include<iostream>
#include<algorithm>
#include<cmath>
using namespace std;
typedef long long ll;
ll n,m,a,b;
ll ceil_div(ll a,ll b){                          //向上取整除法: ceil(a/b) for positive b
    if(b<0){
        a=-a;
        b=-b;
    }
    if(a>=0) return (a+b-1)/b;
    else return a/b;
}
ll floor_div(ll a,ll b){                         //向下取整除法: floor(a/b) for positive b
    if(b<0){
        a=-a;
        b=-b;
    }
    if(a>=0) return a/b;
    else return (a-b+1)/b;
}
bool check(ll x){
    if((a+b)*x>n+m) return false;                //特殊情况：总券数不够
    if(a==b) return (a*x<=n)&&(a*x<=m);          //特殊情况：兑换数量相等
    ll diff=a-b;
    ll lower_bound_k,upper_bound_k;
    if(diff>0){
        lower_bound_k=ceil_div(x*a-m,diff);
        upper_bound_k=floor_div(n-x*b,diff);
    }else{
        ll d=-diff;
        lower_bound_k=ceil_div(x*b-n,d);
        upper_bound_k=floor_div(m-x*a,d);
    }
    lower_bound_k=max(lower_bound_k,0LL);        //k必须在[0, x]范围内
    upper_bound_k=min(upper_bound_k,x);
    return lower_bound_k<=upper_bound_k;
}
int main(){
    //输入
    cin>>n>>m>>a>>b;
    //计算
    ll l=0;
    ll r=(n+m)/(a+b);
    ll ans=0;
    while(l<=r){
        ll mid=l+(r-l)/2;
        if(check(mid)){
            ans=mid;
            l=mid+1;
        }else{
            r=mid-1;
        }
    }
    //输出
    cout<<ans<<endl;
    return 0;
}
```

```c
//P13014 [GESP202506 五级] 最大公因数
#include<iostream>
#include<algorithm>
#include<vector>
using namespace std;
typedef long long ll;
ll gcd(ll a,ll b){
    while(b!=0){
        ll temp=b;
        b=a%b;
        a=temp;
    }
    return a;
}
int main(){
    //输入
    int n,q;
    cin>>n>>q;
    vector<ll> a(n);
    for(int i=0;i<n;++i){
        cin>>a[i];
    }
    //计算
    sort(a.begin(),a.end());
    ll diff_gcd=0;
    for(int i=1;i<n;++i){
        ll diff=a[i]-a[i-1];
        diff_gcd=gcd(diff_gcd,diff);
    }
    for(int i=1;i<=q;++i){
        ll current_val=a[0]+i;
        ll ans=gcd(current_val,diff_gcd);
        //输出
        cout<<ans<<"\n";
    }
    return 0;
}
```

```c
//P14073 [GESP202509 五级] 数字选取

```

```c
//P14074 [GESP202509 五级] 有趣的数字和

```

```c
//P14917 [GESP202512 五级] 数字移动

```

```c
//P14918 [GESP202512 五级] 相等序列

```